### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


# Lecture 12.3: Internal and External Clustering Metrics (Same Data)

Given the same generated dataset, this notebook computes and compares:
- Internal: Silhouette, Davies-Bouldin, Dunn
- External: Rand Index, Purity, Mutual Information


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

rng = np.random.default_rng(5)


def make_data(points_per_cluster=40, noise=0.55):
    centers = np.array([[-2,0],[2,0],[0,2.5]])
    X=[]
    y=[]
    for i,c in enumerate(centers):
        Xi = c + rng.normal(0, noise, size=(points_per_cluster,2))
        X.append(Xi)
        y.extend([i]*points_per_cluster)
    return np.vstack(X), np.array(y)


def kmeans(X, k=3, iters=20):
    idx = rng.choice(len(X), size=k, replace=False)
    C = X[idx].copy()
    for _ in range(iters):
        d = ((X[:,None,:]-C[None,:,:])**2).sum(axis=2)
        y = d.argmin(axis=1)
        C = np.vstack([X[y==j].mean(axis=0) if np.any(y==j) else C[j] for j in range(k)])
    return y


def pairwise_dist(A, B=None):
    B = A if B is None else B
    return np.sqrt(((A[:,None,:]-B[None,:,:])**2).sum(axis=2))


def silhouette_score(X, y):
    D = pairwise_dist(X)
    n = len(X)
    s = np.zeros(n)
    labels = np.unique(y)
    for i in range(n):
        yi = y[i]
        same = np.where(y==yi)[0]
        a = D[i, same[same!=i]].mean() if len(same)>1 else 0
        b = np.inf
        for lbl in labels:
            if lbl==yi:
                continue
            idx = np.where(y==lbl)[0]
            b = min(b, D[i, idx].mean())
        s[i] = (b-a)/max(a,b) if max(a,b)>0 else 0
    return float(np.mean(s))


def davies_bouldin(X, y):
    labels = np.unique(y)
    centroids = np.array([X[y==l].mean(axis=0) for l in labels])
    S = np.array([np.mean(np.linalg.norm(X[y==l]-centroids[i], axis=1)) for i,l in enumerate(labels)])
    M = pairwise_dist(centroids)
    R = np.zeros((len(labels), len(labels)))
    for i in range(len(labels)):
        for j in range(len(labels)):
            if i!=j and M[i,j] > 0:
                R[i,j] = (S[i]+S[j])/M[i,j]
    D = np.max(R, axis=1)
    return float(np.mean(D))


def dunn_index(X, y):
    labels = np.unique(y)
    diam = []
    inter = []
    for l in labels:
        Xi = X[y==l]
        if len(Xi) > 1:
            diam.append(np.max(pairwise_dist(Xi)))
        else:
            diam.append(0)
    for i,li in enumerate(labels):
        for lj in labels[i+1:]:
            inter.append(np.min(pairwise_dist(X[y==li], X[y==lj])))
    return float(np.min(inter) / (np.max(diam)+1e-12))


def rand_index(y_true, y_pred):
    n=len(y_true)
    agree=0
    total=0
    for i in range(n):
        for j in range(i+1,n):
            same_t = y_true[i]==y_true[j]
            same_p = y_pred[i]==y_pred[j]
            agree += int(same_t==same_p)
            total += 1
    return agree/total


def purity(y_true, y_pred):
    labels=np.unique(y_pred)
    total=0
    for l in labels:
        idx = np.where(y_pred==l)[0]
        if len(idx)==0:
            continue
        counts = np.bincount(y_true[idx])
        total += counts.max()
    return total/len(y_true)


def mutual_info(y_true, y_pred):
    n=len(y_true)
    t_labels = np.unique(y_true)
    p_labels = np.unique(y_pred)
    pt = np.array([(y_true==t).mean() for t in t_labels])
    pp = np.array([(y_pred==p).mean() for p in p_labels])
    mi = 0.0
    for i,t in enumerate(t_labels):
        for j,p in enumerate(p_labels):
            pij = np.mean((y_true==t)&(y_pred==p))
            if pij > 0:
                mi += pij*np.log(pij/(pt[i]*pp[j]))
    return float(mi)


def draw(points_per_cluster=40, noise=0.55, k=3):
    X, y_true = make_data(points_per_cluster=points_per_cluster, noise=noise)
    y_pred = kmeans(X, k=k)

    si = silhouette_score(X, y_pred)
    db = davies_bouldin(X, y_pred)
    du = dunn_index(X, y_pred)
    ri = rand_index(y_true, y_pred)
    pu = purity(y_true, y_pred)
    mi = mutual_info(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    ax.scatter(X[:,0], X[:,1], c=y_pred, cmap='tab10', s=16)
    ax.set_title('Predicted clusters on same dataset')
    ax.grid(alpha=0.2)
    plt.show()

    print('Internal metrics')
    print(f'  Silhouette (higher better):      {si:.4f}')
    print(f'  Davies-Bouldin (lower better):   {db:.4f}')
    print(f'  Dunn (higher better):            {du:.4f}')
    print('External metrics')
    print(f'  Rand Index (higher better):      {ri:.4f}')
    print(f'  Purity (higher better):          {pu:.4f}')
    print(f'  Mutual Information (higher):     {mi:.4f}')

widgets.interact(
    draw,
    points_per_cluster=widgets.IntSlider(description='pts/cluster', min=20, max=120, step=5, value=40, continuous_update=False),
    noise=widgets.FloatSlider(description='noise', min=0.1, max=1.2, step=0.05, value=0.55, continuous_update=False),
    k=widgets.IntSlider(description='k', min=2, max=6, value=3, continuous_update=False),
)
